# Hantush-Jacob (1955) — Constant Rate Test Analysis
**Leaky confined aquifer model**

Same as Theis but for aquifers that have a leaky layer above or below them. Fits three parameters: T, S, and B (leakage factor).

---
### How to use
1. Run the **Install & Import** cell once
2. Fill in your values in the **USER INPUTS** cell
3. Click **Run All** (or press `Shift+Enter` through each cell)

In [ ]:
# Install required packages (only needed once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy', 'scipy', 'matplotlib', '-q'])
print('All packages ready.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
print('Imports done.')

---
## ✏️ USER INPUTS — fill in your values here

In [ ]:
# ── YOUR DATA FILE ────────────────────────────────────────────────────────────
# Put the name of your CSV file below (must be in the same folder as this notebook)
# CSV must have two columns:  time (minutes) , drawdown (metres)
# Set to None to run a demo with synthetic data instead
DATA_FILE = 'crt_data.csv'   # <-- change this to your file name, or set to None

# ── TEST PARAMETERS ───────────────────────────────────────────────────────────
Q = 2.5      # Pumping rate            (L/s)
r = 30.0     # Distance to obs well    (metres)

# ── INITIAL GUESSES ───────────────────────────────────────────────────────────
# These are your best estimates before fitting.
# They don't need to be exact — the script will optimise from here.
T_init = 100.0   # Transmissivity guess   (m²/day)
S_init = 1e-4    # Storativity guess      (dimensionless, typically 1e-5 to 1e-3)
B_init = 500.0   # Leakage factor guess   (metres)
                 # Large B (e.g. 5000) = low leakage, similar to Theis
                 # Small B (e.g. 100)  = high leakage, drawdown flattens out

# ── BOURDET SMOOTHING ─────────────────────────────────────────────────────────
L = 0.8      # Smoothing factor (0.8 is standard — leave this unless you know why to change it)

---
## Load data
*(no changes needed below this line)*

In [ ]:
# Unit conversions
_MIN_TO_S  = 60.0
_DAY_TO_S  = 86400.0
_LS_TO_M3S = 1e-3

Q_si      = Q * _LS_TO_M3S
T_init_si = T_init / _DAY_TO_S

# Hantush-Jacob well function W(u, r/B)
def _hantush_scalar(u, rB):
    if u <= 0 or not np.isfinite(u):
        return np.nan
    try:
        val, _ = quad(lambda y: np.exp(-y - rB**2/(4*y))/y, u, np.inf, limit=200)
        return float(val)
    except:
        return np.nan

_hantush_W = np.vectorize(_hantush_scalar, excluded=['rB'])

def hantush_drawdown(t, T, S, B):
    u  = r**2 * S / (4 * T * t)
    rB = r / B
    return (Q_si / (4 * np.pi * T)) * _hantush_W(u, rB=rB)

def load_csv(path):
    data = np.genfromtxt(path, delimiter=',', comments='#', dtype=float)
    if data.ndim == 1:
        raise ValueError('CSV needs two columns: time(min), drawdown(m)')
    mask = np.isfinite(data[:,0]) & np.isfinite(data[:,1]) & (data[:,0] > 0)
    t = data[mask,0]; s = data[mask,1]
    order = np.argsort(t)
    return t[order] * _MIN_TO_S, s[order]

def make_demo_data():
    t_s = np.logspace(np.log10(1*_MIN_TO_S), np.log10(1440*_MIN_TO_S), 40)
    s   = hantush_drawdown(t_s, T_init_si, S_init, B_init)
    rng = np.random.default_rng(42)
    return t_s, np.maximum(s + rng.normal(0, 0.005, 40), 0)

import os
if DATA_FILE is not None and os.path.exists(DATA_FILE):
    t, s_obs = load_csv(DATA_FILE)
    print(f'Loaded {len(t)} rows from {DATA_FILE}')
else:
    if DATA_FILE is not None:
        print(f'File "{DATA_FILE}" not found — running demo instead.')
    else:
        print('No file set — running demo.')
    print('Generating demo data (this may take a few seconds)...')
    t, s_obs = make_demo_data()
    print(f'Demo data: {len(t)} points')

print(f'Time range:     {t[0]/_MIN_TO_S:.2f} – {t[-1]/_MIN_TO_S:.2f} min')
print(f'Drawdown range: {s_obs.min():.4f} – {s_obs.max():.4f} m')

---
## Fit the Hantush-Jacob model

In [ ]:
print('Fitting model — please wait, this may take 10–30 seconds...')

try:
    popt, pcov = curve_fit(
        hantush_drawdown, t, s_obs,
        p0=[T_init_si, S_init, B_init],
        bounds=([1e-10, 1e-12, 1.0], [10.0, 1.0, 1e7]),
        maxfev=50000
    )
    T_fit, S_fit, B_fit = popt
    perr = np.sqrt(np.diag(pcov))
    print('Fitting successful!')
except RuntimeError as e:
    print(f'Warning: fitting did not converge — {e}')
    T_fit, S_fit, B_fit = T_init_si, S_init, B_init
    perr = [np.nan, np.nan, np.nan]

T_fit_day = T_fit * _DAY_TO_S
leakance  = T_fit_day / B_fit**2
s_model   = hantush_drawdown(t, T_fit, S_fit, B_fit)

# Fit statistics
res    = s_obs - s_model
rmse   = np.sqrt(np.mean(res**2))
mae    = np.mean(np.abs(res))
ss_res = np.sum(res**2)
ss_tot = np.sum((s_obs - s_obs.mean())**2)
r2     = 1 - ss_res/ss_tot

print()
print('='*55)
print('  HANTUSH-JACOB SOLUTION — RESULTS')
print('='*55)
print(f'  Pumping rate Q     = {Q} L/s')
print(f'  Distance     r     = {r} m')
print(f'  Fitted T           = {T_fit_day:.4f} ± {perr[0]*_DAY_TO_S:.4f} m²/day')
print(f'  Fitted S           = {S_fit:.2e} ± {perr[1]:.2e}')
print(f'  Fitted B           = {B_fit:.4f} ± {perr[2]:.4f} m')
print(f'  Leakance K\'/b\'    = {leakance:.4e} day⁻¹')
print(f'  T/S (diffusivity)  = {T_fit_day/S_fit:.2e} m²/day')
print(f'  r/B                = {r/B_fit:.4f}  (<<1 means nearly confined)')
print('-'*55)
print('  Model Fit:')
print(f'    R²   = {r2:.4f}  (1.0 = perfect)')
print(f'    RMSE = {rmse:.4f} m')
print(f'    MAE  = {mae:.4f} m')
print('='*55)

---
## Bourdet derivative

In [ ]:
def bourdet_derivative(t, s, L=0.8):
    n    = len(t)
    ln_t = np.log(t)
    deriv = np.full(n, np.nan)
    for i in range(n):
        a = next((j for j in range(i-1,-1,-1) if ln_t[i]-ln_t[j] >= L), None)
        b = next((j for j in range(i+1, n)    if ln_t[j]-ln_t[i] >= L), None)
        if a is not None and b is not None:
            dA = ln_t[i]-ln_t[a]; dB = ln_t[b]-ln_t[i]
            deriv[i] = ((s[i]-s[a])/dA*dB + (s[b]-s[i])/dB*dA) / (dA+dB)
        elif a is not None:
            deriv[i] = (s[i]-s[a]) / (ln_t[i]-ln_t[a])
        elif b is not None:
            deriv[i] = (s[b]-s[i]) / (ln_t[b]-ln_t[i])
    return deriv

d_obs   = bourdet_derivative(t, s_obs,   L=L)
d_model = bourdet_derivative(t, s_model, L=L)

valid = ~(np.isnan(d_obs)|np.isnan(d_model)|(d_obs<=0)|(d_model<=0))
print(f'Bourdet derivative (L={L}): {valid.sum()} valid points out of {len(t)}')

if valid.sum() >= 2:
    dres  = d_obs[valid] - d_model[valid]
    drmse = np.sqrt(np.mean(dres**2))
    dtot  = np.sum((d_obs[valid]-d_obs[valid].mean())**2)
    dr2   = 1 - np.sum(dres**2)/dtot if dtot > 0 else np.nan
    print(f'Derivative R²   = {dr2:.4f}')
    print(f'Derivative RMSE = {drmse:.4f} m')

---
## Diagnostic plots

In [ ]:
t_min = t / _MIN_TO_S

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(
    f'Hantush-Jacob (1955) CRT Analysis — Leaky Confined Aquifer\n'
    f'T = {T_fit_day:.4f} m²/day   S = {S_fit:.2e}   B = {B_fit:.2f} m   '
    f'K\'/b\' = {leakance:.2e} day⁻¹\n'
    f'R² = {r2:.4f}   RMSE = {rmse:.4f} m',
    fontsize=10, fontweight='bold'
)

# 1. Log-log: drawdown + derivative
ax = axes[0,0]
ax.loglog(t_min, s_obs,   'o', color='steelblue', ms=5, label='Observed Δs', zorder=4)
ax.loglog(t_min, s_model, '-', color='firebrick',  lw=2, label='H-J model Δs')
m1 = ~np.isnan(d_obs)   & (d_obs   > 0)
m2 = ~np.isnan(d_model) & (d_model > 0)
if m1.any(): ax.loglog(t_min[m1], d_obs[m1],   's', color='cornflowerblue', ms=5, mfc='none', label=f"Obs Δs' (L={L})")
if m2.any(): ax.loglog(t_min[m2], d_model[m2], '--', color='salmon', lw=1.5, label="Model Δs'")
ax.set_xlabel('Time (min)'); ax.set_ylabel('Δs / Δs\' (m)')
ax.set_title('Log–Log Diagnostic (Bourdet derivative)')
ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)

# 2. Semi-log drawdown
ax = axes[0,1]
ax.semilogx(t_min, s_obs,   'o', color='steelblue', ms=5, label='Observed')
ax.semilogx(t_min, s_model, '-', color='firebrick',  lw=2, label='H-J model')
ax.set_xlabel('Time (min)'); ax.set_ylabel('Drawdown (m)')
ax.set_title('Semi-Log Drawdown')
ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)

# 3. Observed vs modelled
ax = axes[1,0]
ax.scatter(s_model, s_obs, c='steelblue', s=25, alpha=0.75, zorder=3)
lo = min(s_obs.min(), s_model.min()) * 0.85
hi = max(s_obs.max(), s_model.max()) * 1.10
ax.plot([lo,hi],[lo,hi],'k--',lw=1,label='1:1 line')
ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
ax.set_xlabel('Modelled Drawdown (m)'); ax.set_ylabel('Observed Drawdown (m)')
ax.set_title(f'Observed vs Modelled  (R² = {r2:.4f})')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 4. Residuals
ax = axes[1,1]
ax.semilogx(t_min, s_obs-s_model, 'o', color='darkorange', ms=5)
ax.axhline(0, color='k', lw=1, linestyle='--')
ax.set_xlabel('Time (min)'); ax.set_ylabel('Residual obs − model (m)')
ax.set_title(f'Residuals  (RMSE = {rmse:.4f} m)')
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('hantush_crt_analysis.png', dpi=150, bbox_inches='tight')
print('Plot saved as hantush_crt_analysis.png')
plt.show()